<!-- notebook-header -->
# Transformers e BERT

**Modulo:** 05 - Dominios Aplicados / 05B - NLP  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Self-attention, multi-head attention, positional encoding, BERT e fine-tuning.


# 5B_4: Transformers e BERT

**Objetivo**: Compreender a arquitetura Transformer e o modelo BERT que revolucionaram NLP.

**Contexto**: Em 5B_3, vimos que RNNs processam sequencias de forma SEQUENCIAL e sofrem
com vanishing gradient. Attention resolve o gargalo mas ainda depende de recorrencia.
A pergunta crucial: "se attention funciona tao bem, por que nao usar APENAS attention?"
Essa pergunta levou ao paper "Attention is All You Need" (Vaswani et al., 2017).

**Neste notebook:**
1. Limitacoes de RNNs que motivaram Transformers
2. Self-attention e scaled dot-product
3. Multi-head attention
4. Positional encoding
5. Arquitetura completa do Transformer
6. BERT e pre-training (MLM, NSP)
7. Fine-tuning e variantes (RoBERTa, DistilBERT)
8. Exercicios praticos com implementacoes numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
np.random.seed(42)

## 1. De RNNs para Transformers

### Analogia
Imagine ler um livro para escrever um resumo. Uma RNN le PALAVRA POR PALAVRA,
da primeira a ultima -- se o livro tem 1000 paginas, demora 1000 passos.
Um Transformer, por outro lado, le TODAS AS PAGINAS SIMULTANEAMENTE e depois
decide quais sao importantes (via attention). E como ter 1000 leitores, cada
um lendo uma pagina ao mesmo tempo.

### Definicao Formal
Limitacoes de RNNs que Transformers resolvem:
1. **Sequencialidade:** RNN processa t=1, t=2, ... em ORDEM. Nao paraleliza.
2. **Distancia:** Info de t=1 chega em t=100 apos 99 transformacoes (vanishing gradient).
3. **Gargalo:** Em seq2seq, toda info e comprimida num unico vetor h_T.

Transformers usam self-attention: cada posicao "olha" para TODAS as outras
diretamente, sem precisar passar por posicoes intermediarias.

### Por que em ML
Paralelizacao e crucial para treinar em GPUs. RNNs usam <10% da capacidade
de uma GPU moderna. Transformers usam >90% porque TODAS as posicoes sao
processadas simultaneamente. Isso permitiu escalar para bilhoes de parametros.

In [ ]:
# Comparar complexidade RNN vs Transformer
seq_lengths = [10, 50, 100, 500, 1000, 5000]
hidden_size = 512

rnn_sequential = []  # O(T) passos sequenciais
transformer_sequential = []  # O(1) passos sequenciais
rnn_compute = []  # O(T * H^2)
transformer_compute = []  # O(T^2 * H)

for T in seq_lengths:
    rnn_sequential.append(T)
    transformer_sequential.append(1)
    rnn_compute.append(T * hidden_size**2)
    transformer_compute.append(T**2 * hidden_size)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Passos sequenciais
ax = axes[0]
ax.bar(range(len(seq_lengths)), rnn_sequential, width=0.35, label='RNN', color='red', alpha=0.7)
ax.bar([x+0.35 for x in range(len(seq_lengths))], transformer_sequential, width=0.35,
       label='Transformer', color='blue', alpha=0.7)
ax.set_xticks([x+0.175 for x in range(len(seq_lengths))])
ax.set_xticklabels(seq_lengths)
ax.set_xlabel('Comprimento da sequencia')
ax.set_ylabel('Passos sequenciais')
ax.set_title('Paralelizacao: RNN vs Transformer', fontweight='bold')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# Computacao total
ax = axes[1]
ax.plot(seq_lengths, rnn_compute, 'r-o', label='RNN: O(T*H^2)', linewidth=2)
ax.plot(seq_lengths, transformer_compute, 'b-s', label='Transformer: O(T^2*H)', linewidth=2)
ax.axvline(x=hidden_size, color='green', linestyle='--', alpha=0.5, label=f'T=H={hidden_size}')
ax.set_xlabel('Comprimento da sequencia (T)')
ax.set_ylabel('Computacao total')
ax.set_title('Computacao: RNN vs Transformer (H=512)', fontweight='bold')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/rnn_vs_transformer.png', dpi=100, bbox_inches='tight')
plt.show()

print("RNN: T passos sequenciais, Transformer: 1 passo (paralelo)")
print(f"Para T=1000: RNN precisa de 1000 passos, Transformer precisa de 1")
print(f"Computacao: Transformer e O(T^2*H), pior que RNN para T > H")
print(f"Mas: Transformer e PARALELO, entao wall-time e muito menor!")

## 2. Self-Attention (Scaled Dot-Product)

### Analogia
Self-attention e como uma sala de reuniao onde cada pessoa (palavra) faz 3 perguntas:
1. **Query (Q):** "O que estou procurando?" (minha necessidade)
2. **Key (K):** "O que eu ofereco?" (minha identidade)
3. **Value (V):** "Qual e meu conteudo?" (minha informacao)

Cada pessoa compara sua Query com as Keys de TAREFA DO ALUNOS os outros. Quem tiver
Key mais similar recebe mais atencao (peso maior). O output e a media
ponderada dos Values de todos.

### Definicao Formal
Para uma sequencia X de T tokens com dimensao d_model:
- Q = X @ W_Q, K = X @ W_K, V = X @ W_V  (projecoes lineares)
- Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V

A divisao por sqrt(d_k) e crucial: sem ela, para d_k grande, os scores
ficam enormes e softmax satura (gradientes muito pequenos).

### Por que em ML
Self-attention captura relacoes entre QUALQUER par de palavras em O(1) "hops"
(vs O(T) para RNNs). "The cat sat on the mat because IT was tired" -- self-attention
conecta "it" diretamente a "cat", sem passar por "sat", "on", "the", "mat".

In [ ]:
# Self-attention implementado do zero com numpy
def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q: (T_q, d_k), K: (T_k, d_k), V: (T_k, d_v)
    d_k = Q.shape[-1]

    # Scores: Q @ K^T / sqrt(d_k)
    scores = Q @ K.T / np.sqrt(d_k)  # (T_q, T_k)

    # Opcional: aplicar mask (para decoder, evitar ver o futuro)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)

    # Softmax -> pesos de atencao
    attention_weights = softmax(scores, axis=-1)  # (T_q, T_k)

    # Output: media ponderada dos Values
    output = attention_weights @ V  # (T_q, d_v)

    return output, attention_weights

# Exemplo: self-attention em "o gato sentou no tapete"
np.random.seed(42)
T = 5  # 5 tokens
d_model = 8

# Embeddings (simulados)
X = np.random.randn(T, d_model)
palavras = ['o', 'gato', 'sentou', 'no', 'tapete']

# Projecoes Q, K, V (matrizes de peso)
d_k = d_v = 4
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_v) * 0.1

Q = X @ W_Q  # (5, 4)
K = X @ W_K  # (5, 4)
V = X @ W_V  # (5, 4)

output, weights = scaled_dot_product_attention(Q, K, V)

print("Scaled Dot-Product Self-Attention")
print(f"  Input: {T} tokens, d_model={d_model}")
print(f"  Q, K shape: ({T}, {d_k}), V shape: ({T}, {d_v})")
print(f"  Output shape: {output.shape}")
print(f"  Weights shape: {weights.shape}")
print()
print("Pesos de atencao (cada linha soma 1):")
for i, p in enumerate(palavras):
    w_str = ', '.join(f'{w:.3f}' for w in weights[i])
    print(f"  '{p}' olha para: [{w_str}]")

In [ ]:
# Visualizar attention weights como heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Attention heatmap
ax = axes[0]
im = ax.imshow(weights, cmap='Blues', aspect='auto')
ax.set_xticks(range(T))
ax.set_xticklabels(palavras, fontsize=10)
ax.set_yticks(range(T))
ax.set_yticklabels(palavras, fontsize=10)
ax.set_xlabel('Key (de quem recebe atencao)')
ax.set_ylabel('Query (quem esta olhando)')
ax.set_title('Self-Attention Weights', fontweight='bold')
plt.colorbar(im, ax=ax)

# Anotar valores
for i in range(T):
    for j in range(T):
        ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', fontsize=8)

# 2. Por que dividir por sqrt(d_k)
ax = axes[1]
d_ks = [4, 16, 64, 256, 1024]
q = np.random.randn(1, 64)

for d_k_test in d_ks:
    k = np.random.randn(10, d_k_test)
    q_test = np.random.randn(1, d_k_test)

    scores_raw = (q_test @ k.T).flatten()
    scores_scaled = scores_raw / np.sqrt(d_k_test)

    ax.scatter([d_k_test]*10, scores_raw, c='red', alpha=0.3, s=20)
    ax.scatter([d_k_test]*10, scores_scaled, c='blue', alpha=0.3, s=20)

ax.scatter([], [], c='red', label='Sem scaling')
ax.scatter([], [], c='blue', label='Com /sqrt(d_k)')
ax.set_xlabel('d_k (dimensao)')
ax.set_ylabel('Score')
ax.set_title('Efeito do Scaling em Dot-Product', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/self_attention.png', dpi=100, bbox_inches='tight')
plt.show()

print("Sem scaling: scores crescem com sqrt(d_k), softmax satura")
print("Com scaling: scores ficam em range estavel, gradientes saudaveis")

### O que observar sobre Attention Weights na Pratica

Pesos de attention quase uniformes (como no exemplo acima) indicam que o modelo
NAO aprendeu ainda (pesos aleatorios). Apos treinamento, attention se torna
SELETIVA: cada token foca em poucos tokens relevantes, nao em todos igualmente.

### O que concluir sobre Softmax e Temperatura

O "sharpness" dos pesos de attention e controlado pela escala dos scores.
Scores grandes -> softmax afiada (1 token domina). Scores pequenos -> softmax
suave (distribuicao uniforme). Dividir por sqrt(d_k) mantem escala intermediaria.

## 3. Multi-Head Attention

### Analogia
Uma cabeca de attention ve UM tipo de relacao. E como olhar uma foto com
um filtro -- voce ve uma coisa mas perde outras. Multi-head e como olhar
com VARIOS filtros simultaneamente: um ve cores, outro ve bordas, outro
ve texturas. Combinados, dao uma visao muito mais rica.

### Definicao Formal
Multi-head attention com h heads:
- head_i = Attention(X @ W_Q^i, X @ W_K^i, X @ W_V^i)
- MultiHead(X) = Concat(head_1, ..., head_h) @ W_O

Cada head tem suas proprias projecoes W_Q, W_K, W_V de dimensao d_k = d_model / h.
A concatenacao e projetada de volta para d_model via W_O.

### Por que em ML
Em BERT-base (12 heads), cada head aprende relacoes diferentes:
- Head 1: sujeito-verbo ("gato SENTOU")
- Head 3: adjetivo-substantivo ("gato PRETO")
- Head 7: coreference ("gato...ELE")
- Head 10: posicao relativa (vizinhos proximos)

Isso da ao modelo uma representacao MULTIFACETADA de cada token.

In [ ]:
# Multi-Head Attention com numpy
def multi_head_attention(X, num_heads, d_model):
    # X: (T, d_model)
    T = X.shape[0]
    d_k = d_model // num_heads
    d_v = d_k

    all_heads = []
    all_weights = []

    for h in range(num_heads):
        # Projecoes independentes para cada head
        W_Q = np.random.randn(d_model, d_k) * 0.1
        W_K = np.random.randn(d_model, d_k) * 0.1
        W_V = np.random.randn(d_model, d_v) * 0.1

        Q = X @ W_Q
        K = X @ W_K
        V = X @ W_V

        head_output, head_weights = scaled_dot_product_attention(Q, K, V)
        all_heads.append(head_output)
        all_weights.append(head_weights)

    # Concatenar todos os heads
    concat = np.concatenate(all_heads, axis=-1)  # (T, d_model)

    # Projecao de saida
    W_O = np.random.randn(d_model, d_model) * 0.1
    output = concat @ W_O  # (T, d_model)

    return output, all_weights

# Teste
np.random.seed(42)
T = 5
d_model = 16
num_heads = 4
X = np.random.randn(T, d_model)

output, head_weights = multi_head_attention(X, num_heads, d_model)

print(f"Multi-Head Attention ({num_heads} heads)")
print(f"  Input: ({T}, {d_model})")
print(f"  d_k per head: {d_model // num_heads}")
print(f"  Output: {output.shape}")
print()

# Visualizar que cada head olha para coisas diferentes
fig, axes = plt.subplots(1, num_heads, figsize=(16, 3.5))
for h in range(num_heads):
    ax = axes[h]
    im = ax.imshow(head_weights[h], cmap='Blues', aspect='auto')
    ax.set_title(f'Head {h+1}', fontweight='bold', fontsize=10)
    ax.set_xticks(range(T))
    ax.set_xticklabels(palavras, fontsize=8, rotation=45)
    ax.set_yticks(range(T))
    ax.set_yticklabels(palavras, fontsize=8)
plt.suptitle('Cada head aprende padroes de atencao DIFERENTES', fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/multi_head.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Positional Encoding

### Analogia
Transformers processam TAREFA DO ALUNOS os tokens simultaneamente, sem nocao de ordem.
E como dar cartas de baralho sem numerar -- ninguem sabe a ordem.
Positional encoding e "numerar as cartas" com um codigo especial (sin/cos)
que permite ao modelo descobrir posicoes RELATIVAS.

### Definicao Formal
PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

Propriedade chave: PE(pos+k) pode ser expressa como transformacao LINEAR de PE(pos).
Isso permite ao modelo aprender relacoes de posicao RELATIVA.

### Por que em ML
Sem positional encoding, "gato come peixe" e "peixe come gato" seriam
IDENTICOS para o Transformer (mesmo bag of embeddings). PE preserva
a informacao de ordem sem sacrificar paralelizacao.

In [ ]:
# Positional Encoding com numpy
def positional_encoding(max_len, d_model):
    PE = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]  # (max_len, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

    PE[:, 0::2] = np.sin(position * div_term)  # dimensoes pares
    PE[:, 1::2] = np.cos(position * div_term)  # dimensoes impares

    return PE

# Gerar e visualizar
max_len = 50
d_model = 64
PE = positional_encoding(max_len, d_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Heatmap do positional encoding
ax = axes[0]
im = ax.imshow(PE[:30, :32], cmap='RdBu', aspect='auto')
ax.set_xlabel('Dimensao')
ax.set_ylabel('Posicao')
ax.set_title('Positional Encoding (sin/cos)', fontweight='bold')
plt.colorbar(im, ax=ax)

# 2. Ondas sin/cos para dimensoes especificas
ax = axes[1]
positions = range(max_len)
for dim in [0, 2, 4, 8, 16]:
    ax.plot(positions, PE[:, dim], label=f'dim {dim}', linewidth=1.5)
ax.set_xlabel('Posicao')
ax.set_ylabel('Valor')
ax.set_title('Frequencias Diferentes por Dimensao', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/positional_encoding.png', dpi=100, bbox_inches='tight')
plt.show()

# Verificar propriedade: posicoes proximas sao mais similares
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Similaridade entre posicoes (cosseno):")
print(f"  pos 0 vs pos 1:  {cosine_sim(PE[0], PE[1]):.4f}  (proximas)")
print(f"  pos 0 vs pos 5:  {cosine_sim(PE[0], PE[5]):.4f}  (medias)")
print(f"  pos 0 vs pos 25: {cosine_sim(PE[0], PE[25]):.4f} (distantes)")
print(f"  pos 0 vs pos 49: {cosine_sim(PE[0], PE[49]):.4f} (muito distantes)")

### O que observar sobre Alternativas ao Positional Encoding

Alem de sin/cos (PE absoluto), existem alternativas:
- **Learned PE:** aprender embeddings de posicao como parametros (BERT usa isso)
- **Relative PE:** codificar DISTANCIA entre tokens, nao posicao absoluta (T5)
- **RoPE (Rotary):** rotacionar embeddings com base na posicao (LLaMA, GPT-NeoX)

### O que concluir sobre a Limitacao de Comprimento

PE fixo limita o comprimento maximo. BERT original: max 512 tokens.
Para textos mais longos, opcoes incluem: chunking (dividir em pedacos),
Longformer (attention local + global), ou RoPE (extrapola melhor).

### Conexao com outros notebooks sobre Dados Sequenciais

Positional encoding resolve o mesmo problema que RNNs (5B_3): capturar
informacao de POSICAO. Mas enquanto RNNs codificam posicao implicitamente
(processar em ordem), PE codifica explicitamente (somar vetor ao embedding).

## 5. Arquitetura Completa do Transformer

### Definicao Formal
Cada bloco do Transformer Encoder tem:
1. **Multi-Head Self-Attention** + Residual Connection + Layer Norm
2. **Feed-Forward Network** (2 camadas lineares com ReLU) + Residual + Layer Norm

Encoder = N blocos empilhados (BERT-base: N=12)

### Por que em ML
- **Residual connections:** permitem gradientes fluir sem degradacao (como ResNets, 5A_1)
- **Layer Norm:** estabiliza treinamento normalizando dentro de cada sample
- **FFN:** adiciona nao-linearidade e aumenta capacidade do modelo

### O que observar sobre a Estrutura
O output de cada bloco tem a MESMA dimensao do input (d_model=768 em BERT-base).
Isso permite empilhar blocos arbitrariamente -- cada bloco refina a representacao.

In [ ]:
# Transformer Encoder Block com numpy
def layer_norm(x, eps=1e-6):
    mean = np.mean(x, axis=-1, keepdims=True)
    std = np.std(x, axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def feed_forward(x, d_model, d_ff):
    # FFN(x) = max(0, x @ W1 + b1) @ W2 + b2
    W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)
    b1 = np.zeros(d_ff)
    W2 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)
    b2 = np.zeros(d_model)

    hidden = np.maximum(0, x @ W1 + b1)  # ReLU
    return hidden @ W2 + b2

def transformer_encoder_block(X, num_heads, d_model, d_ff):
    # 1. Multi-Head Self-Attention + Residual + LayerNorm
    attn_output, weights = multi_head_attention(X, num_heads, d_model)
    X = layer_norm(X + attn_output)  # Residual + LayerNorm

    # 2. Feed-Forward + Residual + LayerNorm
    ffn_output = feed_forward(X, d_model, d_ff)
    X = layer_norm(X + ffn_output)  # Residual + LayerNorm

    return X, weights

# Teste: processar sequencia por 1 bloco
np.random.seed(42)
T = 5
d_model = 16
d_ff = 64
num_heads = 4

X = np.random.randn(T, d_model)
output, weights = transformer_encoder_block(X, num_heads, d_model, d_ff)

print("Transformer Encoder Block")
print(f"  Input: ({T}, {d_model})")
print(f"  Num heads: {num_heads}")
print(f"  FFN dim: {d_ff}")
print(f"  Output: {output.shape}")
print(f"  Input == Output dimensao: {X.shape == output.shape}")
print()

# Empilhar N blocos
N_layers = 3
X_current = X.copy()
for layer in range(N_layers):
    np.random.seed(layer * 10)  # reprodutibilidade
    X_current, _ = transformer_encoder_block(X_current, num_heads, d_model, d_ff)
    print(f"  Layer {layer+1} output shape: {X_current.shape}, norm: {np.linalg.norm(X_current):.3f}")

print(f"\n{N_layers} blocos empilhados, dimensao preservada!")

### O que observar sobre Residual Connections e Layer Norm

Residual connections sao ESSENCIAIS: sem elas, empilhar 12+ blocos causa
vanishing gradient. LayerNorm estabiliza as ativacoes dentro de cada sample.
A ordem importa: Pre-LN (norm antes de attention) e mais estavel que Post-LN
(norm depois), e e o padrao em modelos modernos (GPT-3, LLaMA).

### O que concluir sobre o Feed-Forward Network

O FFN em cada bloco expande de d_model para d_ff (4x maior) e volta.
Essa expansao cria um "bottleneck invertido" que permite transformacoes
mais ricas. Estudos mostram que FFN funciona como "memoria" que armazena
padroes aprendidos, enquanto attention funciona como "roteamento".

### Por que em ML a Profundidade Importa

Cada bloco refina progressivamente as representacoes:
- Camadas iniciais: features sintaticas (POS tags, dependencias locais)
- Camadas medias: features semanticas (significado, relacoes)
- Camadas finais: features especificas da tarefa

### Conexao com outros notebooks sobre Deep Learning

A ideia de empilhar blocos identicos vem de ResNets (5A_1): cada bloco
adiciona um "refinamento incremental" via residual connection.
Tanto em CNNs quanto em Transformers, profundidade = representacoes melhores.

## 6. BERT (Bidirectional Encoder Representations from Transformers)

### Analogia
Imagine um teste de cloze: "O ___ sentou no tapete". Humanos facilmente
preenchem "gato" porque entendemos o contexto de AMBOS os lados. BERT
faz exatamente isso: mascara 15% das palavras e aprende a preve-las
usando contexto BIDIRECIONAL. Isso e diferente de GPT que so olha para a esquerda.

### Definicao Formal
BERT usa dois objetivos de pre-training:
1. **MLM (Masked Language Model):** mascarar 15% dos tokens e prever
   - 80% substituidos por [MASK]
   - 10% substituidos por token aleatorio
   - 10% mantidos originais
2. **NSP (Next Sentence Prediction):** prever se sentenca B segue A

### Por que em ML
Pre-training bidirecional captura contexto COMPLETO. Fine-tuning transfere
esse conhecimento para tarefas especificas com poucos dados. BERT-base
(110M parametros) superou modelos especializados em 11 benchmarks de NLP.

In [ ]:
# Simular MLM (Masked Language Model) com numpy
def create_mlm_data(token_ids, mask_prob=0.15, vocab_size=100, special_tokens=4):
    # token_ids: lista de IDs (excluindo special tokens 0-3)
    masked_ids = token_ids.copy()
    labels = [-100] * len(token_ids)  # -100 = ignore

    for i, tid in enumerate(token_ids):
        if tid < special_tokens:  # skip [CLS], [SEP], [MASK], [PAD]
            continue

        if np.random.random() < mask_prob:
            labels[i] = tid  # guardar original como label

            r = np.random.random()
            if r < 0.8:
                masked_ids[i] = 2  # [MASK] token
            elif r < 0.9:
                masked_ids[i] = np.random.randint(special_tokens, vocab_size)
            # else: manter original (10%)

    return masked_ids, labels

# Simular tokenizacao e MLM
np.random.seed(42)
vocab = {
    '[CLS]': 0, '[SEP]': 1, '[MASK]': 2, '[PAD]': 3,
    'o': 4, 'gato': 5, 'sentou': 6, 'no': 7, 'tapete': 8,
    'azul': 9, 'e': 10, 'dormiu': 11
}
idx2word = {v: k for k, v in vocab.items()}

# Frase: [CLS] o gato sentou no tapete azul [SEP]
original = [0, 4, 5, 6, 7, 8, 9, 1]
masked, labels = create_mlm_data(original, mask_prob=0.3)  # prob alta p/ demo

print("Masked Language Model (MLM)")
print("=" * 50)
print(f"Original: {[idx2word[t] for t in original]}")
print(f"Masked:   {[idx2word.get(t, f'[{t}]') for t in masked]}")
print(f"Labels:   {[idx2word.get(l, 'ignore') for l in labels]}")
print()
print("O modelo deve prever as palavras mascaradas usando contexto bidirecional")
print("80% [MASK], 10% random, 10% original -- evita o modelo depender do token [MASK]")

### O que observar sobre a Genialidade do MLM

MLM e genial porque:
1. Nao precisa de dados rotulados (self-supervised!)
2. Forca contexto BIDIRECIONAL (diferente de language models unidirecionais)
3. A mascara 80/10/10 evita que o modelo "trapaceie" procurando [MASK]

### O que concluir sobre Pre-Training como Compressao de Conhecimento

Pre-training comprime BILHOES de paginas de texto em parametros do modelo.
BERT "sabe" gramatica, semantica, e fatos do mundo porque VIU tudo no
pre-training. Fine-tuning DESBLOQUEIA esse conhecimento para tarefas especificas.

### Conexao com outros notebooks sobre Aprendizado Nao Supervisionado

MLM e uma forma de aprendizado auto-supervisionado (como autoencoders, 5C).
A tarefa pretext (prever tokens mascarados) forca o modelo a aprender
representacoes uteis sem labels humanos.

## 7. Fine-Tuning BERT

### Por que em ML
Pre-training e CARO (milhares de GPU-hours). Fine-tuning e BARATO (horas).
A ideia: BERT aprendeu "entender linguagem" no pre-training. Fine-tuning
adapta esse conhecimento para SUA tarefa especifica.

### O que observar sobre o Token [CLS]
Em BERT, o primeiro token e sempre [CLS] (classificacao). O hidden state
desse token na ultima camada e usado como representacao da SEQUENCIA INTEIRA.
Para classificacao, basta adicionar uma camada linear sobre [CLS].

### O que concluir sobre Transfer Learning em NLP
BERT inaugurou a era de transfer learning em NLP (similar a ImageNet em visao).
O pipeline moderno: pre-train em corpus ENORME -> fine-tune em dados especificos.
Isso democratizou NLP: nao precisa mais de BILHOES de dados para cada tarefa.

In [ ]:
# Simular fine-tuning de BERT para classificacao
np.random.seed(42)

# Simular output do BERT (hidden states da ultima camada)
# Na pratica: model.encode(text) -> (T, 768)
batch_size = 6
seq_len = 10
hidden_size = 16  # BERT-base usa 768

# Dados simulados: 3 positivos, 3 negativos
bert_outputs = np.random.randn(batch_size, seq_len, hidden_size)
labels = np.array([1, 1, 1, 0, 0, 0])

# Extrair [CLS] token (posicao 0)
cls_representations = bert_outputs[:, 0, :]  # (batch, hidden_size)

# Classification head: uma camada linear + softmax
W_cls = np.random.randn(hidden_size, 2) * 0.01
b_cls = np.zeros(2)

logits = cls_representations @ W_cls + b_cls  # (batch, 2)
probs = softmax(logits, axis=-1)

print("Fine-tuning BERT para Classificacao")
print("=" * 50)
print(f"  BERT output: ({batch_size}, {seq_len}, {hidden_size})")
print(f"  CLS representation: {cls_representations.shape}")
print(f"  Classification head: Linear({hidden_size}, 2)")
print()
for i in range(batch_size):
    pred = "positivo" if probs[i, 1] > 0.5 else "negativo"
    real = "positivo" if labels[i] == 1 else "negativo"
    print(f"  Sample {i}: P(pos)={probs[i,1]:.3f}, pred={pred}, real={real}")
print()
print("Antes do fine-tuning: predicoes aleatorias (pesos random)")
print("Apos fine-tuning: modelo aprende a classificar usando features do BERT")

In [ ]:
# Comparar variantes de BERT
variantes = {
    'BERT-base': {'params': 110, 'layers': 12, 'hidden': 768, 'heads': 12, 'speed': 1.0},
    'BERT-large': {'params': 340, 'layers': 24, 'hidden': 1024, 'heads': 16, 'speed': 0.3},
    'RoBERTa': {'params': 125, 'layers': 12, 'hidden': 768, 'heads': 12, 'speed': 1.0},
    'DistilBERT': {'params': 66, 'layers': 6, 'hidden': 768, 'heads': 12, 'speed': 1.6},
    'ALBERT': {'params': 12, 'layers': 12, 'hidden': 768, 'heads': 12, 'speed': 0.9},
    'ELECTRA': {'params': 110, 'layers': 12, 'hidden': 768, 'heads': 12, 'speed': 1.0},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Parametros
ax = axes[0]
names = list(variantes.keys())
params = [v['params'] for v in variantes.values()]
colors = ['steelblue', 'navy', 'green', 'orange', 'red', 'purple']
bars = ax.barh(names, params, color=colors, alpha=0.7)
ax.set_xlabel('Parametros (milhoes)')
ax.set_title('Tamanho dos Modelos', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
for bar, p in zip(bars, params):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{p}M', va='center', fontsize=9)

# Speed vs params
ax = axes[1]
for name, v in variantes.items():
    ax.scatter(v['params'], v['speed'], s=100, zorder=5)
    ax.annotate(name, (v['params'], v['speed']), fontsize=8,
               xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('Parametros (M)')
ax.set_ylabel('Velocidade relativa')
ax.set_title('Trade-off: Tamanho vs Velocidade', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/bert_variants.png', dpi=100, bbox_inches='tight')
plt.show()

print("Variantes de BERT:")
print(f"  DistilBERT: 40% menor, 60% mais rapido, 97% da accuracy")
print(f"  ALBERT: 89% menor (parameter sharing), mesma accuracy")
print(f"  RoBERTa: mesmo tamanho, melhor treino -> melhor accuracy")
print(f"  ELECTRA: substitui MLM por discriminador, mais eficiente")

## 8. Exercicios Praticos

### Exercicio 1: Implementar Scaled Dot-Product Attention
Implemente attention do zero e verifique que os pesos somam 1.

In [ ]:
# PRATICA - Exercicio 1: Scaled Dot-Product Attention

def my_attention(Q, K, V):
    # Q: (T_q, d_k), K: (T_k, d_k), V: (T_k, d_v)
    # Retorna: output (T_q, d_v), weights (T_q, T_k)

    d_k = Q.shape[-1]

    # TAREFA DO ALUNO: calcular scores = Q @ K^T / sqrt(d_k)
    scores = None

    # TAREFA DO ALUNO: aplicar softmax nas linhas
    weights = None

    # TAREFA DO ALUNO: calcular output = weights @ V
    output = None

    return output, weights

# Teste
Q = np.array([[1.0, 0.0], [0.0, 1.0]])
K = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
V = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])

# output, weights = my_attention(Q, K, V)
# print(f"Weights soma: {weights.sum(axis=1)}")  # Deve ser [1, 1]
print("Implemente my_attention acima!")

In [ ]:
# SOLUCAO - Exercicio 1: Scaled Dot-Product Attention
def my_attention_sol(Q, K, V):
    d_k = Q.shape[-1]

    # Scores
    scores = Q @ K.T / np.sqrt(d_k)

    # Softmax
    exp_s = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = exp_s / exp_s.sum(axis=-1, keepdims=True)

    # Output
    output = weights @ V

    return output, weights

Q = np.array([[1.0, 0.0], [0.0, 1.0]])
K = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
V = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])

output, weights = my_attention_sol(Q, K, V)

print("Scaled Dot-Product Attention - SOLUCAO")
print("=" * 50)
print(f"Q shape: {Q.shape}, K shape: {K.shape}, V shape: {V.shape}")
print(f"Weights:\n{weights}")
print(f"Weights soma por linha: {weights.sum(axis=1)}")
print(f"Output:\n{output}")
print()
print("Q[0]=[1,0] -> mais atencao em K[0]=[1,0] (similar)")
print("Q[1]=[0,1] -> mais atencao em K[1]=[0,1] (similar)")

### Exercicio 2: Positional Encoding Manual
Calcule PE para posicoes 0 e 1 com d_model=4 e verifique as propriedades.

In [ ]:
# PRATICA - Exercicio 2: Positional Encoding
# Calcular PE manualmente para d_model=4, posicoes 0 e 1

d_model = 4
# PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
# PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

# TAREFA DO ALUNO: calcular PE para pos=0
# dim 0 (2i=0): sin(0 / 10000^(0/4)) = sin(0) = ?
# dim 1 (2i+1=1): cos(0 / 10000^(0/4)) = cos(0) = ?
# dim 2 (2i=2): sin(0 / 10000^(2/4)) = ?
# dim 3 (2i+1=3): cos(0 / 10000^(2/4)) = ?
pe_0 = None  # np.array([?, ?, ?, ?])

# TAREFA DO ALUNO: calcular PE para pos=1
pe_1 = None

print("Calcule PE para pos=0 e pos=1 manualmente!")

In [ ]:
# SOLUCAO - Exercicio 2: Positional Encoding
d_model = 4

# Para pos=0:
# dim 0: sin(0 / 10000^(0/4)) = sin(0) = 0
# dim 1: cos(0 / 10000^(0/4)) = cos(0) = 1
# dim 2: sin(0 / 10000^(2/4)) = sin(0) = 0
# dim 3: cos(0 / 10000^(2/4)) = cos(0) = 1
pe_0 = np.array([0, 1, 0, 1])

# Para pos=1:
div_0 = 10000 ** (0/4)  # = 1
div_2 = 10000 ** (2/4)  # = 100
# dim 0: sin(1/1) = sin(1) = 0.841
# dim 1: cos(1/1) = cos(1) = 0.540
# dim 2: sin(1/100) = sin(0.01) = 0.010
# dim 3: cos(1/100) = cos(0.01) = 1.000
pe_1 = np.array([np.sin(1/div_0), np.cos(1/div_0), np.sin(1/div_2), np.cos(1/div_2)])

# Verificar com funcao
PE_check = positional_encoding(2, 4)

print("Positional Encoding Manual - SOLUCAO")
print("=" * 50)
print(f"PE(pos=0): {pe_0}")
print(f"  Verificacao: {PE_check[0]}")
print(f"  Match: {np.allclose(pe_0, PE_check[0])}")
print()
print(f"PE(pos=1): [{', '.join(f'{v:.4f}' for v in pe_1)}]")
print(f"  Verificacao: [{', '.join(f'{v:.4f}' for v in PE_check[1])}]")
print(f"  Match: {np.allclose(pe_1, PE_check[1])}")
print()
print(f"Distancia pos0-pos1: {np.linalg.norm(pe_0 - pe_1):.4f}")
print("Dim 0,1 variam RAPIDO (freq alta), dim 2,3 variam DEVAGAR (freq baixa)")

### Exercicio 3: MLM Prediction
Dado um contexto mascarado, use attention para prever a palavra.

In [ ]:
# PRATICA - Exercicio 3: MLM Prediction
# Simular predicao de palavra mascarada

# Contexto: "o [MASK] sentou no tapete"
# Candidatos: gato, cachorro, livro, mesa
candidatos = {
    'gato': np.array([0.8, 0.2, 0.9, 0.1]),
    'cachorro': np.array([0.7, 0.3, 0.8, 0.15]),
    'livro': np.array([0.1, 0.9, 0.2, 0.8]),
    'mesa': np.array([0.2, 0.8, 0.3, 0.7])
}

# Contexto embedding (media das palavras vizinhas)
contexto = np.array([0.5, 0.3, 0.7, 0.2])  # "sentou no tapete"

# TAREFA DO ALUNO: calcular similaridade cosseno entre contexto e cada candidato
# TAREFA DO ALUNO: aplicar softmax para obter probabilidades
# TAREFA DO ALUNO: qual palavra e mais provavel?

print("Calcule a predicao MLM!")

In [ ]:
# SOLUCAO - Exercicio 3: MLM Prediction
candidatos = {
    'gato': np.array([0.8, 0.2, 0.9, 0.1]),
    'cachorro': np.array([0.7, 0.3, 0.8, 0.15]),
    'livro': np.array([0.1, 0.9, 0.2, 0.8]),
    'mesa': np.array([0.2, 0.8, 0.3, 0.7])
}
contexto = np.array([0.5, 0.3, 0.7, 0.2])

# Similaridade cosseno
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sims = {}
for nome, emb in candidatos.items():
    sims[nome] = cosine_similarity(contexto, emb)

# Softmax para probabilidades
scores = np.array(list(sims.values()))
exp_s = np.exp(scores * 5)  # temperature scaling
probs = exp_s / exp_s.sum()

print("MLM Prediction - SOLUCAO")
print("=" * 50)
print(f'Contexto: "o [MASK] sentou no tapete"')
print()
for i, (nome, sim) in enumerate(sims.items()):
    barra = '#' * int(probs[i] * 40)
    print(f"  {nome:10s}: sim={sim:.3f}, P={probs[i]:.3f} {barra}")
print()
melhor = list(sims.keys())[np.argmax(probs)]
print(f"Predicao: '{melhor}' (semanticamente mais proximo do contexto)")
print("BERT usa contexto BIDIRECIONAL: tanto 'o' quanto 'sentou no tapete'")

## 9. Erros Comuns e Armadilhas

### Erro 1: Nao usar causal mask no decoder
**Problema:** Decoder ve tokens futuros durante treinamento (data leakage).
**Solucao:** Aplicar mascara triangular que bloqueia posicoes futuras.

### Erro 2: Learning rate muito alto para fine-tuning
**Problema:** LR > 5e-5 para BERT causa "catastrophic forgetting" do pre-training.
**Solucao:** Usar LR pequeno (2e-5 a 5e-5) com warm-up linear.

### Erro 3: Nao escalar embeddings por sqrt(d_model)
**Problema:** Embeddings tem magnitude pequena, PE domina a representacao.
**Solucao:** Multiplicar embeddings por sqrt(d_model) antes de somar PE.

### Erro 4: Ignorar attention mask para padding
**Problema:** Tokens de [PAD] participam do attention, corrompendo representacoes.
**Solucao:** Passar attention_mask que exclui tokens de padding.

### Erro 5: Fine-tuning com poucos dados sem regularizacao
**Problema:** BERT-base tem 110M parametros, overfita facilmente com <1000 exemplos.
**Solucao:** Usar dropout, weight decay, early stopping, ou DistilBERT (menor).

### O que observar sobre o Impacto do Transformer na IA Moderna

O paper "Attention is All You Need" (2017) e o mais citado da decada em ML.
A arquitetura Transformer nao so dominou NLP, mas expandiu para:
- Visao: ViT, Swin Transformer (5A_5)
- Audio: Whisper, AudioLM
- Proteinas: AlphaFold 2
- Multimodal: CLIP, DALL-E, GPT-4V

### O que concluir sobre a Revolucao do Pre-Training

Antes de BERT, cada tarefa de NLP era treinada DO ZERO. BERT mostrou que
pre-training + fine-tuning e superior: o modelo aprende "entender linguagem"
primeiro, depois adapta para tarefas especificas. Isso reduziu drasticamente
a quantidade de dados rotulados necessarios.

### Conexao com outros notebooks sobre Transfer Learning

BERT e para NLP o que ImageNet pre-training e para visao (5A_2).
O conceito e identico: treinar em tarefa generica -> adaptar para tarefa especifica.
Em ambos os casos, as primeiras camadas aprendem features genericas (sintaxe em BERT,
bordas em CNNs) e as ultimas aprendem features especificas da tarefa.

### O que observar sobre Self-Attention vs Cross-Attention

Self-attention: Q, K, V vem da MESMA sequencia (Encoder de BERT)
Cross-attention: Q vem do decoder, K e V vem do encoder (Transformer original)
Self-attention captura relacoes INTERNAS; cross-attention conecta duas sequencias.

### O que concluir sobre Eficiencia de Transformers

Transformers sao O(T^2) em memoria e computacao. Para T=1024, isso significa
~1M operacoes de attention. Alternativas: Longformer (janela local + global),
Linformer (projecao linear), FlashAttention (IO-aware). Na pratica,
FlashAttention e o padrao atual para treinamento eficiente.

### Conexao com outros notebooks sobre Interpretabilidade

Attention weights de Transformers sao usadas para interpretabilidade (4_3),
mas com cuidado: "attention != explanation". Pesos de attention mostram
QUANTO cada token contribuiu, mas nao necessariamente POR QUE.
BertViz e uma ferramenta popular para visualizar attention patterns.

### O que observar sobre Encoder vs Decoder vs Encoder-Decoder

- **Encoder-only (BERT):** bidirecional, bom para compreensao (classificacao, NER)
- **Decoder-only (GPT):** unidirecional, bom para geracao (texto, codigo)
- **Encoder-Decoder (T5, BART):** bom para transformacao (traducao, resumo)

### O que concluir sobre a Escolha de Modelo na Pratica

Na pratica, a escolha depende da tarefa e recursos:
- Classificacao: BERT ou DistilBERT (rapido, preciso)
- Geracao: GPT (autogressivo, flexivel)
- Traducao/Resumo: T5 ou BART (encoder-decoder)
- Recursos limitados: DistilBERT ou ALBERT

### Conexao com outros notebooks sobre Escalabilidade

A escalabilidade de Transformers levou a "scaling laws" (5B_5): performance
melhora previsivelmente com mais dados, parametros e computacao.
Isso motivou LLMs como GPT-3 (175B) e GPT-4 -- temas do proximo notebook.

### Conexao com outros notebooks sobre Otimizacao de Transformers

Treinar Transformers requer tecnicas especificas de otimizacao (3_1):
- AdamW (Adam com weight decay desacoplado)
- Learning rate warm-up (evita instabilidade no inicio)
- Cosine annealing (decay suave)
- Gradient accumulation (batch grande em GPU pequena)

### O que observar sobre Tokenizacao Subword em BERT

BERT usa WordPiece tokenization: palavras raras sao divididas em subwords.
"embeddings" -> "em" + "##bed" + "##ding" + "##s". Isso resolve o problema
de vocabulario aberto (OOV) e conecta com FastText subwords (5B_2).

### O que concluir sobre o Ecosistema Hugging Face

Hugging Face democratizou o acesso a Transformers: milhares de modelos
pre-treinados disponiveis gratuitamente. Na pratica, raramente se treina
BERT do zero -- usa-se modelos pre-treinados e faz-se fine-tuning.

### O que concluir sobre BERT para Portugues

BERTimbau (BERT pre-treinado em portugues brasileiro) supera BERT multilingual
em tarefas de NLP em portugues. Escolher o modelo certo para o IDIOMA
dos dados e tao importante quanto escolher a arquitetura certa.

### Conexao com outros notebooks sobre Word Embeddings

BERT produz embeddings CONTEXTUAIS: a representacao de "banco" muda dependendo
se o contexto e financeiro ou de rio (limitacao de Word2Vec/GloVe, 5B_2).
Cada camada de BERT produz embeddings diferentes -- camadas finais sao melhores
para tarefas semanticas, camadas iniciais para tarefas sintaticas.

### Conexao com outros notebooks sobre Classificacao

Fine-tuning de BERT para classificacao conecta com classificadores classicos (2_4):
em vez de features manuais (TF-IDF + SVM), BERT APRENDE features e classifica
end-to-end. Isso tipicamente supera pipelines tradicionais por larga margem.

### Conexao com outros notebooks sobre Modelos Generativos

BERT e encoder-only (compreensao). GPT (5B_5) e decoder-only (geracao).
T5 unifica ambos tratando TODA tarefa como text-to-text: "translate English
to German: The house is wonderful" -> "Das Haus ist wunderbar".

## Resumo e Proximos Passos

### Hierarquia de Conceitos

```
Transformer (Attention is All You Need, 2017)
  |
  +-- Self-Attention
  |     |-- Q, K, V: projecoes lineares do input
  |     |-- Scores = Q @ K^T / sqrt(d_k)
  |     +-- Output = softmax(scores) @ V
  |
  +-- Multi-Head Attention
  |     |-- h heads independentes
  |     |-- Cada head: d_k = d_model / h
  |     +-- Concat + projecao W_O
  |
  +-- Positional Encoding
  |     |-- sin/cos com frequencias diferentes
  |     +-- Permite aprender posicoes relativas
  |
  +-- Transformer Block
  |     |-- MHA + Residual + LayerNorm
  |     +-- FFN + Residual + LayerNorm
  |
  +-- BERT (2018)
  |     |-- Encoder-only, bidirecional
  |     |-- Pre-training: MLM + NSP
  |     +-- Fine-tuning: CLS token + linear head
  |
  +-- Variantes
        |-- RoBERTa: treino otimizado
        |-- DistilBERT: compressao (knowledge distillation)
        |-- ALBERT: parameter sharing
        +-- ELECTRA: discriminador + gerador
```

### Conexoes entre Notebooks

| Conceito | Notebook | Relacao |
|----------|----------|---------|
| RNNs/Attention | 5B_3 | Motivacao para Transformers |
| Residual connections | 5A_1 | Mesmo principio em CNNs |
| Transfer learning | 5A_2 | ImageNet pre-training analogia |
| Interpretabilidade | 4_3 | Attention como ferramenta |
| Regularizacao | 4_1, 4_2 | Dropout, weight decay |
| Vision Transformers | 5A_5 | Transformer para imagens |
| LLMs | 5B_5 | Escalar Transformers |

### Checklist de Competencias

- [ ] Sei explicar por que Transformers substituiram RNNs
- [ ] Consigo implementar scaled dot-product attention
- [ ] Entendo multi-head attention e por que multiplas heads
- [ ] Sei como positional encoding preserva informacao de ordem
- [ ] Consigo descrever um Transformer Encoder Block completo
- [ ] Entendo MLM e NSP como objetivos de pre-training do BERT
- [ ] Sei como fazer fine-tuning do BERT para classificacao
- [ ] Consigo comparar variantes (RoBERTa, DistilBERT, ALBERT)

### Proximos Passos
No proximo notebook (5B_5), veremos como ESCALAR Transformers para
bilhoes de parametros criou os Large Language Models (GPT, LLaMA, etc.)
e como isso mudou fundamentalmente a IA.